## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set your project path
PROJECT_PATH = '/content/drive/MyDrive/vindr-spinexr'

import os
os.chdir(PROJECT_PATH)
print(f"Working directory: {os.getcwd()}")

## Step 2: Check GPU

In [ ]:
!nvidia-smi

## Step 3: Install Dependencies

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")

In [ ]:
# Install detectron2
!pip install 'git+https://github.com/facebookresearch/detectron2.git@4841e70ee48da72c32304f9ebf98138c2a70048d'

In [ ]:
# Install other dependencies
!pip install timm pycocotools scikit-learn pandas pydot

## Step 4: Verify Installation

In [ ]:
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer
from detectron2.config import get_cfg

print(f"Detectron2 version: {detectron2.__version__}")
print("✓ Detectron2 installed successfully!")

## Step 5: Verify Data Files ✓

In [ ]:
import os
import pandas as pd
import glob

# Check data structure
print("Checking data files...")
print(f"✓ Train annotations: {os.path.exists('data/annotations/train.csv')}")
print(f"✓ Train images dir: {os.path.exists('data/train_pngs')}")
print(f"✓ Config file: {os.path.exists('spine/configs/sparsercnn_improved.yaml')}")
print(f"✓ Pretrained weights: {os.path.exists('pretrained/r101_100pro_3x_model.pth')}")

# Count images
num_images = len(glob.glob('data/train_pngs/*.png'))
print(f"\n✓ Found {num_images} training images")

# Load and check annotations
train_df = pd.read_csv('data/annotations/train.csv')
print(f"✓ Total annotations: {len(train_df)}")
print(f"✓ Unique images: {train_df['image_id'].nunique()}")
print(f"\nLesion distribution:")
print(train_df['lesion_type'].value_counts())

print("\n🎉 All data verified! Ready to train!")

## Step 6: Start Training 🚀

In [ ]:
# Train the model
!python spine/train_net.py \
    --num-gpus 1 \
    --config-file spine/configs/sparsercnn_improved.yaml \
    OUTPUT_DIR outputs/sparsercnn_improved

## Step 7: Monitor Training (Optional)

In [ ]:
# View training logs
!tail -n 50 outputs/sparsercnn_improved/log.txt

## Step 8: Evaluate Final Model

In [ ]:
# Evaluate the final model
!python spine/train_net.py \
    --eval-only \
    --num-gpus 1 \
    --config-file spine/configs/sparsercnn_improved.yaml \
    MODEL.WEIGHTS outputs/sparsercnn_improved/model_final.pth

## Step 9: Compare with Paper Table 4 Results

In [ ]:
import json

# Load metrics from our training
with open('outputs/sparsercnn_improved/metrics.json', 'r') as f:
    metrics = [json.loads(line) for line in f]

# Get final mAP@0.5 (bbox/AP50 in COCO metrics)
final_metrics = metrics[-1]
our_map50 = final_metrics.get('bbox/AP50', 0)

# Paper Table 4 - Detection Models Comparison
# Columns: LT2, LT4, LT6, LT8, LT10, LT11, LT13, mAP@0.5
paper_results = {
    "Faster R-CNN": {
        "LT2": 22.66, "LT4": 35.99, "LT6": 49.24, "LT8": 31.68,
        "LT10": 65.22, "LT11": 51.68, "LT13": 2.16, "mAP@0.5": 31.83
    },
    "RetinaNet": {
        "LT2": 14.53, "LT4": 25.35, "LT6": 41.67, "LT8": 32.14,
        "LT10": 65.49, "LT11": 51.85, "LT13": 5.30, "mAP@0.5": 28.09
    },
    "EfficientDet": {
        "LT2": 17.05, "LT4": 24.19, "LT6": 42.69, "LT8": 35.18,
        "LT10": 61.85, "LT11": 52.53, "LT13": 2.45, "mAP@0.5": 28.73
    },
    "Sparse R-CNN (Paper)": {
        "LT2": 20.09, "LT4": 32.67, "LT6": 48.16, "LT8": 45.32,
        "LT10": 72.20, "LT11": 49.30, "LT13": 5.41, "mAP@0.5": 33.15
    }
}

# Display Table 4 Reproduction
print("=" * 95)
print("TABLE 4: Detection Model Comparison (Reproducing Paper Results)")
print("=" * 95)
print(f"{'Detector':<25} {'LT2':>7} {'LT4':>7} {'LT6':>7} {'LT8':>7} {'LT10':>7} {'LT11':>7} {'LT13':>7} {'mAP@0.5':>10}")
print("-" * 95)

for model, scores in paper_results.items():
    print(f"{model:<25} {scores['LT2']:>7.2f} {scores['LT4']:>7.2f} {scores['LT6']:>7.2f} {scores['LT8']:>7.2f} "
          f"{scores['LT10']:>7.2f} {scores['LT11']:>7.2f} {scores['LT13']:>7.2f} {scores['mAP@0.5']:>10.2f}")

print("-" * 95)
print(f"{'OUR Sparse R-CNN':<25} {'N/A':>7} {'N/A':>7} {'N/A':>7} {'N/A':>7} "
      f"{'N/A':>7} {'N/A':>7} {'N/A':>7} {our_map50:>10.2f}")
print("=" * 95)

# Analysis
paper_best = paper_results["Sparse R-CNN (Paper)"]["mAP@0.5"]
improvement = our_map50 - paper_best

print(f"\n📊 ANALYSIS:")
print(f"   Paper's best model: Sparse R-CNN @ {paper_best:.2f} mAP@0.5")
print(f"   Our Sparse R-CNN:   {our_map50:.2f} mAP@0.5")
print(f"   Difference:         {improvement:+.2f} mAP")
print(f"   Target goal:        36-38 mAP@0.5")

if our_map50 >= 36.0:
    print("\n🎉 EXCELLENT! Reached target range (36-38 mAP@0.5)!")
elif our_map50 > paper_best:
    print(f"\n✅ GOOD! Beat paper's {paper_best:.2f} baseline!")
    print(f"   Still {36.0 - our_map50:.2f} mAP below target 36.0")
else:
    print(f"\n⚠️ Below paper baseline. Need more training/tuning.")

print("\n💡 NOTE: LT2-LT13 are lesion types (Disc narrowing, Osteophytes, etc.)")
print("   We only show overall mAP@0.5 - per-class metrics available in log.txt")

## Step 10: Results Summary

In [ ]:
print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)
print("\nResults saved to Google Drive:")
print(f"  {PROJECT_PATH}/outputs/sparsercnn_improved/")
print("\nFiles:")
print("  • model_final.pth - Trained model weights")
print("  • metrics.json - Training metrics")
print("  • log.txt - Full training log")
print("\n✓ All results automatically synced to your Google Drive!")
print("\nYou can close this notebook - results are saved!")